# OmniVoice 快速上手

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice_ZH.ipynb)

本笔记本（Notebook）演示了 [OmniVoice](https://github.com/k2-fsa/OmniVoice) 的基本用法。OmniVoice 是一个支持 600 多种语言的大规模多语言 Zero-Shot（零样本）TTS 模型。

**目录：**
1. 安装 (Installation)
2. 选项 A — Gradio 演示界面（交互式 Web UI，无需编写代码）
3. 选项 B — Python API 接口
   - 3.1 加载模型
   - 3.2 声音克隆 (Voice Cloning)
   - 3.3 声音设计 (Voice Design)
   - 3.4 自动声音选择 (Auto Voice)

## 1. 安装

Colab 已经预先配置了兼容的 PyTorch + CUDA 环境，因此我们只需要安装 OmniVoice 库。

In [ ]:
!pip install omnivoice

## 2. 选项 A — Gradio 演示界面

启动一个带临时公网访问链接的交互式 Web 界面。使用 `--share` 标志会生成一个公网 URL，方便您从任何浏览器直接访问演示页面。

> **如果您更倾向于直接在代码中使用 Python API，可以跳过此步并前往下方的“选项 B”。**

In [ ]:
!omnivoice-demo --share

## 3. 选项 B — Python API 接口

### 3.1 加载模型

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 声音克隆 (Voice Cloning)

基于一段简短（3-10秒）的参考音频克隆指定的说话人音色。您可以直接上传您自己的 `ref.wav` 或者使用任何音频文件。

`ref_text`（参考音频对应的文本内容）是可选的 —— 如果省略此参数，模型将自动调用内置的 Whisper ASR 进行转录。

In [ ]:
from google.colab import files

print("请上传您的参考音频文件 (支持 wav/mp3/flac 格式):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"已上传音频文件: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="您好，这是一次零样本声音克隆的测试。",
    ref_audio=ref_audio_path,
    # ref_text="此处可以手动提供参考音频的文本",  # 可选参数
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 声音设计 (Voice Design)

无需提供参考音频，仅需输入文本并以词汇描述您想要的音色属性。

支持描述的属性包括：性别、年龄、音调、风格（如耳语/低语）、英文口音、中文方言等。全部支持的指令列表请参考 [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md)。

In [ ]:
audio = model.generate(
    text="您好，这是一次零样本声音设计的测试。",
    instruct="女，低音调，耳语",  # 也可以使用英文：instruct="female, low pitch, whisper"
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 自动声音选择 (Auto Voice)

直接由模型自动配置生成适合的高品质声音，无需提供参考音频或风格指令。

In [ ]:
audio = model.generate(
    text="这是一段由模型自动选择声音并生成的语音。",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))